# Stellar tracers in a DM halo with `stracers`

This notebook reproduces the check in the original `tracer_triaxial.ipynb`.
It samples an isotropic Hernquist halo with `stracers.halos.sample_hernquist`, assigns weights for a more concentrated
stellar tracer, and compares the weighted density profile with the input one.

To use a real snapshot, replace the halo construction with arrays read from your
simulation: positions [kpc], velocities [km/s], potential [(km/s)^2], particle
mass [Msun] and IDs. For example, with Gadget files:

```python
from pygadgetreader import readsnap
pos  = readsnap(snap, 'pos', 'dm'); vel = readsnap(snap, 'vel', 'dm')
pot  = readsnap(snap, 'pot', 'dm'); ids = readsnap(snap, 'pid', 'dm')
mass = readsnap(snap, 'mass', 'dm') * 1e10
```


In [ ]:
import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import stracers
from stracers.halos import sample_hernquist   # toy equilibrium halo, in place of a snapshot
from stracers.plotting import plot_diagnostics

In [ ]:
rng = np.random.default_rng(0)
n, M, a = 200_000, 1e10, 10.0
pos, vel, pot = sample_hernquist(n, M, a, seed=rng)
mass = M / n
ids = np.arange(1, n + 1)

## Weights for a Hernquist tracer with a = 2 kpc

In [ ]:
nu = stracers.get_profile("hernquist", a=2.0)
stellar_mass = 1e8
res = stracers.compute_weights(pos, vel, pot, mass, nu, stellar_mass=stellar_mass, rmax=300)
print("sum of stellar mass:", np.sum(res.weights * mass))

In [ ]:
fig = plot_diagnostics(res)

## Recovered density profile

In [ ]:
r = np.linalg.norm(pos, axis=1)
edges = np.logspace(-0.5, np.log10(250), 30)
rc = np.sqrt(edges[1:] * edges[:-1])
vol = 4 / 3 * np.pi * np.diff(edges**3)

rho_dm = np.histogram(r, edges, weights=np.full(n, mass))[0] / vol
rho_st = np.histogram(r, edges, weights=res.weights * mass)[0] / vol

fig, ax = plt.subplots(figsize=(6, 5))
ax.loglog(rc, rho_dm, label="DM")
ax.loglog(rc, rho_st, "o", ms=4, label="weighted DM (tracers)")
ax.loglog(rc, stellar_mass * nu(rc), "k--", label="input tracer profile")
ax.set_xlabel("r [kpc]")
ax.set_ylabel(r"$\rho\ [M_\odot\,\mathrm{kpc}^{-3}]$")
ax.legend();

## Following the tracers to another snapshot

Weights belong to particle IDs. Save them once, then map them onto any later
snapshot.

In [ ]:
fname = os.path.join(tempfile.mkdtemp(), "weights_hernquist_a2.h5")
stracers.save_weights(fname, ids, res.weights, stellar_mass=stellar_mass, profile="hernquist a=2")
ids_w, w, attrs = stracers.load_weights(fname)

ids_later = rng.permutation(ids)          # e.g. IDs read from a later snapshot
w_later = stracers.match_weights(ids_w, w, ids_later)
print(attrs, np.allclose(np.sum(w_later * mass), stellar_mass))